**13/08/2026** -- Inline descriptive statistics for results section *Fire-related PM2.5 throughout Africa*

Using Hu et al. fire PM2.5 data.

In [1]:
# Sys.setenv(PROJ_DATA = "/home/users/cho00/miniconda3/envs/ppca/share/proj")

In [2]:
library(dplyr)
library(readr)
library(tidyr)
# library(sf)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
SCRATCH_DIR <- Sys.getenv("SCRATCH_DIR")
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_2000_2023.csv")
# NAT_BOUNDS_PATH     <- file.path(SCRATCH_DIR, 
#                          "data/spatial/nat_boundaries",
#                          "WB_countries_Admin0_10m")
# western_sahara_path <- file.path(SCRATCH_DIR, 
#                          "data/spatial/nat_boundaries",
#                          "western_sahara/gadm41_ESH_0.shp")

In [4]:
df <- read_csv(DATA_PATH)

Rows: 994412 Columns: 74
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (63): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
# nat_bounds <- sf::read_sf(NAT_BOUNDS_PATH) |> 
#     filter(CONTINENT == "Africa")

In [6]:
# esh_bounds <- sf::read_sf(western_sahara_path)

##### “Average fire PM2.5 concentrations during 2000-2017 varied from X to Y across Africa (1st-99th percentiles)”

2000-2017:

In [9]:
df |> 
    filter(year <= 2017) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE), 
        .groups = "drop"
    ) |> 
    summarise(
        fire_PM25_min = min(fire_PM25, na.rm = TRUE),
        fire_PM25_p01 = quantile(fire_PM25, 0.01, na.rm = TRUE),
        fire_PM25_p99 = quantile(fire_PM25, 0.99, na.rm = TRUE),
        fire_PM25_max = max(fire_PM25, na.rm = TRUE),
    )   

fire_PM25_min,fire_PM25_p01,fire_PM25_p99,fire_PM25_max
<dbl>,<dbl>,<dbl>,<dbl>
0.1680087,0.2947301,17.06127,23.47585


2000-2022:

In [8]:
df |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE), 
        .groups = "drop"
    ) |> 
    summarise(
        fire_PM25_min = min(fire_PM25, na.rm = TRUE),
        fire_PM25_p01 = quantile(fire_PM25, 0.01, na.rm = TRUE),
        fire_PM25_p99 = quantile(fire_PM25, 0.99, na.rm = TRUE),
        fire_PM25_max = max(fire_PM25, na.rm = TRUE),
    )    

fire_PM25_min,fire_PM25_p01,fire_PM25_p99,fire_PM25_max
<dbl>,<dbl>,<dbl>,<dbl>
0.1700488,0.267224,16.21229,22.07518


##### “Fire PM2.5 accounted for X% of total PM2.5 on average (range: X to Y%, Supplementary Figure S4)”

Note: Hu et al. doesn't provide their estimates of all-source PM -- we will use Xu et al.'s all source and Hu fire-specific PM, and see if looks comparable to the 19% average of Xu et al.

In [26]:
# Create fire frac
df <- df |> 
    mutate(fire_frac_hu = fire_PM25_hu / total_PM25)

2000-2017

In [30]:
# Plain average and range
df |> 
    filter(year <= 2017) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_frac = mean(fire_frac_hu, na.rm = TRUE), 
        .groups = "drop"
    ) |> 
    pull(fire_frac) |>
    summary() * 100

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.521   2.251   4.610  11.804  19.210  73.057 

In [34]:
# Population-weighted average
df |> 
    filter(year <= 2017) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_frac = mean(fire_frac_hu, na.rm = TRUE), 
        pop_count = mean(pop_count, na.rm = TRUE),
        .groups = "drop"
    ) |> 
    mutate(pw_fire_frac = fire_frac * pop_count) |>
    summarise(
        pw_fire_frac = sum(pw_fire_frac, na.rm = TRUE) / sum(pop_count, na.rm = TRUE)
    ) |>
    pull(pw_fire_frac) * 100

[1] 13.04984

2000-2022 (note this is actually 2000-2019 because Xu et al. all-source PM only goes up to 2019)

In [38]:
# Plain average and range
df |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_frac = mean(fire_frac_hu, na.rm = TRUE), 
        .groups = "drop"
    ) |> 
    pull(fire_frac) |>
    summary() * 100

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
 0.5069  2.3166  4.7184 11.5758 18.6829 69.9013    2300 

In [36]:
# Population-weighted average
df |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |> 
    summarise(
        fire_frac = mean(fire_frac_hu, na.rm = TRUE), 
        pop_count = mean(pop_count, na.rm = TRUE),
        .groups = "drop"
    ) |> 
    mutate(pw_fire_frac = fire_frac * pop_count) |>
    summarise(
        pw_fire_frac = sum(pw_fire_frac, na.rm = TRUE) / sum(pop_count, na.rm = TRUE)
    ) |>
    pull(pw_fire_frac) * 100

[1] 12.69591

##### “X% of the continent’s land area had fire PM2.5 concentrations above 5 µg/m3 ... Fire PM2.5 concentrations above 10 µg/m3 and 20 µg/m3 occurred across X% and Y% of land area, respectively”

2000-2017

In [41]:
thresholds <- c(5, 10, 20)

df |> 
    filter(year <= 2017) |>
    group_by(lon, lat) |>
    summarise(
        fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE),
        .groups = "drop"
    ) |> 
    crossing(threshold = thresholds) |>
    group_by(threshold) |> 
    summarise(
        N_grids = n(),
        N_above = sum(fire_PM25 > threshold, na.rm = TRUE),
        N_below = sum(fire_PM25 <= threshold, na.rm = TRUE),
        pct_above = 100 * N_above / N_grids,
        pct_below = 100 * N_below / N_grids,
        .groups = "drop"
    )

threshold,N_grids,N_above,N_below,pct_above,pct_below
<dbl>,<int>,<int>,<int>,<dbl>,<dbl>
5,41430,10639,30791,25.6794593,74.32054
10,41430,3869,37561,9.3386435,90.66136
20,41430,79,41351,0.1906831,99.80932


2000-2022

In [42]:
thresholds <- c(5, 10, 20)

df |> 
    filter(year <= 2022) |>
    group_by(lon, lat) |>
    summarise(
        fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE),
        .groups = "drop"
    ) |> 
    crossing(threshold = thresholds) |>
    group_by(threshold) |> 
    summarise(
        N_grids = n(),
        N_above = sum(fire_PM25 > threshold, na.rm = TRUE),
        N_below = sum(fire_PM25 <= threshold, na.rm = TRUE),
        pct_above = 100 * N_above / N_grids,
        pct_below = 100 * N_below / N_grids,
        .groups = "drop"
    )

threshold,N_grids,N_above,N_below,pct_above,pct_below
<dbl>,<int>,<int>,<int>,<dbl>,<dbl>
5,41453,10145,31308,24.47350011,75.52650
10,41453,3679,37774,8.87511157,91.12489
20,41453,29,41424,0.06995875,99.93004


##### “Population-weighted average concentrations were greatest in central African countries (e.g., Democratic Republic of Congo X µg/m3, Angola Y µg/m3) and lowest in northern African countries (e.g., Morocco X µg/m3, Algeria Y µg/m3) and Cabo Verde (X µg/m3)”

2000-2017:

In [ ]:
pw_avg_fire_PM25_country <- df %>% 
    filter(year <= 2017) %>% 
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        across( c("fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) %>% 
    mutate(
        pw_avg_fire_PM25_gt_5 = pw_avg_fire_PM25 >= 5 # exceeds WHO 5μgm-3 annual mean guideline
    ) %>% 
    group_by(country) %>% 
    summarise(
        region = first(region),
        pw_avg_fire_PM25 = mean(pw_avg_fire_PM25),
        n_years_gt_5 = sum(pw_avg_fire_PM25_gt_5),
        avg_fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE)    # plain average
    )

In [48]:
pw_avg_fire_PM25_country %>% arrange(desc(pw_avg_fire_PM25))

country,region,pw_avg_fire_PM25,n_years_gt_5,avg_fire_PM25
<chr>,<chr>,<dbl>,<int>,<dbl>
"Congo, Rep. of",Central Africa,13.1959163,18,9.8002865
"Congo, Democratic Republic of",Central Africa,12.1405858,18,11.9390991
Angola,Central Africa,11.2902189,18,11.0704478
Central African Republic,Central Africa,11.1589506,18,10.4227674
Burundi,Eastern Africa,9.0671719,18,9.0079043
Rwanda,Eastern Africa,8.3989101,18,8.1526627
Zambia,Eastern Africa,7.1635519,17,7.7587551
Gabon,Central Africa,5.8221839,15,6.0001367
Uganda,Eastern Africa,5.7460985,14,5.6617374


##### “In X of the 52 countries, population-weighted annual average fire PM2.5 concentrations alone exceeded the WHO air quality guideline threshold for total PM2.5 in every year between 2000 and 2017”

In [87]:
# No. of countries where annual mean fire PM2.5 exceeds 5μgm-3 every year for 2000-2017
pw_avg_fire_PM25_country %>%
    count(n_years_gt_5 == 18)  # all 18 years

n_years_gt_5 == 18,n
<lgl>,<int>
FALSE,46
TRUE,6


2000-2022:

In [11]:
pw_avg_fire_PM25_country <- df %>% 
    filter(year <= 2022) %>% 
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
        region = recode(region, "Middle Africa" = "Central Africa")
    ) %>% 
    group_by(country, year) %>% 
    summarise(
        region = first(region),
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        across( c("fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) %>% 
    mutate(
        pw_avg_fire_PM25_gt_5 = pw_avg_fire_PM25 >= 5 # exceeds WHO 5μgm-3 annual mean guideline
    ) %>% 
    group_by(country) %>% 
    summarise(
        region = first(region),
        pw_avg_fire_PM25 = mean(pw_avg_fire_PM25),
        n_years_gt_5 = sum(pw_avg_fire_PM25_gt_5),
        avg_fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE)    # plain average
    )

In [12]:
pw_avg_fire_PM25_country %>% arrange(desc(pw_avg_fire_PM25))

country,region,pw_avg_fire_PM25,n_years_gt_5,avg_fire_PM25
<chr>,<chr>,<dbl>,<int>,<dbl>
"Congo, Rep. of",Central Africa,12.0909188,23,9.5579475
"Congo, Democratic Republic of",Central Africa,11.6297472,23,11.6486025
Central African Republic,Central Africa,10.9394975,23,10.2146685
Angola,Central Africa,10.5263460,23,10.6107114
Burundi,Eastern Africa,8.5486191,22,8.4580711
Rwanda,Eastern Africa,7.7693154,22,7.5543012
Zambia,Eastern Africa,6.9120039,20,7.5825917
Gabon,Central Africa,5.8343411,19,6.0152648
Uganda,Eastern Africa,5.5959484,16,5.4711407


In [14]:
# No. countries w/ avg over 5:
print(nrow(pw_avg_fire_PM25_country |> filter(pw_avg_fire_PM25 > 5)))
pw_avg_fire_PM25_country |> filter(pw_avg_fire_PM25 > 5)

[1] 12


country,region,pw_avg_fire_PM25,n_years_gt_5,avg_fire_PM25
<chr>,<chr>,<dbl>,<int>,<dbl>
Angola,Central Africa,10.526346,23,10.610711
Burundi,Eastern Africa,8.548619,22,8.458071
Central African Republic,Central Africa,10.939498,23,10.214669
"Congo, Democratic Republic of",Central Africa,11.629747,23,11.648602
"Congo, Rep. of",Central Africa,12.090919,23,9.557948
Gabon,Central Africa,5.834341,19,6.015265
Malawi,Eastern Africa,5.456871,14,5.491994
Rwanda,Eastern Africa,7.769315,22,7.554301
South Sudan,Eastern Africa,5.533827,17,5.920382


##### “In X of the 52 countries, population-weighted annual average fire PM2.5 concentrations alone exceeded the WHO air quality guideline threshold for total PM2.5 in every year between 2000 and 2022"

In [91]:
# No. of countries where annual mean fire PM2.5 exceeds 5μgm-3 every year for 2000-2022
pw_avg_fire_PM25_country %>%
    count(n_years_gt_5 == 23)  # all 23 years

n_years_gt_5 == 23,n
<lgl>,<int>
FALSE,48
TRUE,4


##### “Population-weighted fire PM2.5 concentrations were higher in rural areas (X µg/m3) than urban (Y µg/m3)”

2000-2017:

In [94]:
df |> 
    filter(year <= 2017) |> 
    filter(!is.na(urban_rural_cat)) |> 
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
    ) |> 
    group_by(urban_rural_cat, year) |> 
    summarise(
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) |> 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) |> 
    group_by(urban_rural_cat) |> 
    summarise(
        pw_avg_fire_PM25 = mean(pw_avg_fire_PM25),
        .groups = "drop"
    )

urban_rural_cat,pw_avg_fire_PM25
<chr>,<dbl>
rural,4.060461
urban,3.122631


2000-2022:

In [95]:
df |> 
    filter(year <= 2022) |> 
    filter(!is.na(urban_rural_cat)) |> 
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
    ) |> 
    group_by(urban_rural_cat, year) |> 
    summarise(
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) |> 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) |> 
    group_by(urban_rural_cat) |> 
    summarise(
        pw_avg_fire_PM25 = mean(pw_avg_fire_PM25),
        .groups = "drop"
    )

urban_rural_cat,pw_avg_fire_PM25
<chr>,<dbl>
rural,3.849481
urban,2.992364


##### “Rural areas had higher fire PM2.5 than urban areas in X out of 48 countries.”

2000-2017

In [122]:
# Population-weighted average fire PM2.5 by country and urban/rural category
pw_avg_fire_PM25_country_ur <- df %>%
    filter(year <= 2017) %>% 
    filter(!is.na(urban_rural_cat)) %>%
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
    ) %>% 
    group_by(country, year, urban_rural_cat) %>% 
    summarise(
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        across( c("fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) %>% 
    group_by(country, urban_rural_cat) %>% 
    summarise(
        pw_avg_fire_PM25 = mean(pw_avg_fire_PM25),
        avg_fire_PM25 = mean(fire_PM25_hu, na.rm = TRUE),   # plain average
        .groups = "drop"
    )

In [123]:
# Pivot to wide-form
pw_avg_fire_PM25_country_ur_wide <- pw_avg_fire_PM25_country_ur %>% 
    pivot_wider(
        id_cols = c("country"),
        names_from = "urban_rural_cat",
        values_from = "pw_avg_fire_PM25",
        names_prefix = "pw_avg_fire_PM25_"
    ) %>% 
    mutate(rural_gt_urban = pw_avg_fire_PM25_rural > pw_avg_fire_PM25_urban)
pw_avg_fire_PM25_country_ur_wide

country,pw_avg_fire_PM25_rural,pw_avg_fire_PM25_urban,rural_gt_urban
<chr>,<dbl>,<dbl>,<lgl>
Algeria,0.5149474,0.5383062,FALSE
Angola,11.6718375,9.6000327,TRUE
Benin,3.9765453,4.0782330,FALSE
Botswana,2.1556832,2.0144284,TRUE
Burkina Faso,1.4001448,1.2004735,TRUE
Burundi,8.8239720,9.8796821,FALSE
Cabo Verde,0.3453503,0.2928691,TRUE
Cameroon,5.1654428,4.8258400,TRUE
Central African Republic,11.0709651,11.4967801,FALSE


In [124]:
# Countries where rural fire PM2.5 exceeds urban fire PM2.5
pw_avg_fire_PM25_country_ur_wide |> count(rural_gt_urban)

rural_gt_urban,n
<lgl>,<int>
FALSE,20
TRUE,28
NA,4


In [125]:
# Countries where urban fire PM2.5 exceeds rural fire PM2.5
pw_avg_fire_PM25_country_ur_wide |> filter(!rural_gt_urban)

country,pw_avg_fire_PM25_rural,pw_avg_fire_PM25_urban,rural_gt_urban
<chr>,<dbl>,<dbl>,<lgl>
Algeria,0.5149474,0.5383062,FALSE
Benin,3.9765453,4.0782330,FALSE
Burundi,8.8239720,9.8796821,FALSE
Central African Republic,11.0709651,11.4967801,FALSE
"Congo, Democratic Republic of",12.0621661,12.4555902,FALSE
"Congo, Rep. of",9.6748855,14.7130220,FALSE
Ghana,4.7216310,4.9201556,FALSE
Kenya,1.4803001,1.7506002,FALSE
Libya,1.1350344,1.2068853,FALSE


2000-2022:

In [135]:
# Population-weighted average fire PM2.5 by country and urban/rural category
pw_avg_fire_PM25_country_ur <- df %>%
    filter(year <= 2022) %>% 
    filter(!is.na(urban_rural_cat)) %>%
    mutate(
        pw_fire_PM25 = fire_PM25_hu * pop_count,
    ) %>% 
    group_by(country, year, urban_rural_cat) %>% 
    summarise(
        across( c("pw_fire_PM25", "pop_count"), ~ sum(.x, na.rm = TRUE) ),
        across( c("fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    mutate(
        pw_avg_fire_PM25 = pw_fire_PM25 / pop_count
    ) %>% 
    group_by(country, urban_rural_cat) %>% 
    summarise(
        pw_avg_fire_PM25    = mean(pw_avg_fire_PM25),
        avg_fire_PM25       = mean(fire_PM25_hu, na.rm = TRUE),   # plain average
        n_years_averaged    = n(),
        .groups = "drop"
    )

In [138]:
# Comoros, Equatorial Guinea, Lesotho, South Sudan have 2 years of 'urban' grid cells
# with the second-generation WorldPop data
pw_avg_fire_PM25_country_ur |> filter(n_years_averaged < 23)

country,urban_rural_cat,pw_avg_fire_PM25,avg_fire_PM25,n_years_averaged
<chr>,<chr>,<dbl>,<dbl>,<int>
Comoros,urban,0.6263113,0.6263113,2
Equatorial Guinea,urban,4.2320536,4.2500895,2
Lesotho,urban,0.7070847,0.7070847,2
South Sudan,urban,6.7947409,7.1000819,2
eSwatini,urban,1.5892183,1.5892183,11


In [140]:
# Pivot to wide-form
pw_avg_fire_PM25_country_ur_wide <- pw_avg_fire_PM25_country_ur %>% 
    pivot_wider(
        id_cols = c("country"),
        names_from = "urban_rural_cat",
        values_from = "pw_avg_fire_PM25",
        names_prefix = "pw_avg_fire_PM25_"
    ) %>% 
    mutate(rural_gt_urban = pw_avg_fire_PM25_rural > pw_avg_fire_PM25_urban)
pw_avg_fire_PM25_country_ur_wide

country,pw_avg_fire_PM25_rural,pw_avg_fire_PM25_urban,rural_gt_urban
<chr>,<dbl>,<dbl>,<lgl>
Algeria,0.4738842,0.4855039,FALSE
Angola,10.9368097,9.1060848,TRUE
Benin,3.5752921,3.7518145,FALSE
Botswana,2.0101499,1.8575519,TRUE
Burkina Faso,1.2930129,1.1253229,TRUE
Burundi,8.2431155,9.2200840,FALSE
Cabo Verde,0.3081674,0.2610138,TRUE
Cameroon,4.9085052,4.6614496,TRUE
Central African Republic,10.7923386,11.4606959,FALSE


In [128]:
# Countries where rural fire PM2.5 exceeds urban fire PM2.5
pw_avg_fire_PM25_country_ur_wide |> count(rural_gt_urban)

rural_gt_urban,n
<lgl>,<int>
FALSE,20
TRUE,32


In [139]:
# Countries where urban fire PM2.5 exceeds rural fire PM2.5
pw_avg_fire_PM25_country_ur_wide |> filter(!rural_gt_urban)

country,pw_avg_fire_PM25_rural,pw_avg_fire_PM25_urban,rural_gt_urban
<chr>,<dbl>,<dbl>,<lgl>
Algeria,0.4738842,0.4855039,FALSE
Benin,3.5752921,3.7518145,FALSE
Burundi,8.2431155,9.2200840,FALSE
Central African Republic,10.7923386,11.4606959,FALSE
"Congo, Rep. of",9.4356542,13.2648274,FALSE
Equatorial Guinea,4.1840118,4.2320536,FALSE
Ghana,4.3396395,4.5487494,FALSE
Kenya,1.3391109,1.6924964,FALSE
Liberia,3.4244266,3.4317318,FALSE
